# Vietnamese Product Reviews — Download & Preprocess

Step 1 of the Naive Bayes vs SVM comparison project. Dataset:
[tuannguyenvananh/vietnamese-text-classification-dataset](https://www.kaggle.com/datasets/tuannguyenvananh/vietnamese-text-classification-dataset)
(Vietnamese e-commerce product reviews, 3-class sentiment: 0 = Negative, 1 = Neutral, 2 = Positive).

(Third dataset for this project — see `Personal Note.md` for why the first two, a single job
posting page and then the UIT-ViSFD aspect-based sentiment dataset, were dropped.)

This revision does two things differently from the first draft:

1. **Investigates each noise category on real data before deciding what to fix** — abbreviations,
   missing diacritics, emoji/icons, glued-together words. Some are safely fixable, some aren't, and
   the difference matters enough to show, not just assert. Every category gets a real before/after
   sample from this corpus, not a made-up one.
2. **Tokenizes with `underthesea`, not `nltk`** — real Vietnamese word segmentation instead of
   syllable splitting (see the "uy tín" finding in Step 0 below for a concrete example of why this
   matters). `underthesea` is known to be slow on large corpora (~2 hours on the 184K-article news
   corpus in `260106_TextPreprocessingwithNLP`), so it gets benchmarked on a small sample first,
   before committing to running it on the whole thing.

## Imports

In [1]:
import os
import re
import string
import time
import unicodedata

import kagglehub
import pandas as pd
from underthesea import word_tokenize

## Download Raw Data

The source file ships as a single headerless CSV (`label,comment` with no header row — the first
data row was getting silently read as a column header before this was caught), and no train/test
split of its own, so both are handled explicitly below.

In [2]:
RAW_DIR = os.path.join("..", "data", "raw")
PROCESSED_DIR = os.path.join("..", "data", "processed")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

dataset_dir = kagglehub.dataset_download("tuannguyenvananh/vietnamese-text-classification-dataset")
df = pd.read_csv(os.path.join(dataset_dir, "train.csv"), header=None, names=["label", "comment"])
df["comment"] = df["comment"].fillna("")
df.to_csv(os.path.join(RAW_DIR, "train.csv"), index=False)

print(df.shape)
print(df["label"].value_counts().sort_index())
df.head()

Label meaning, confirmed by reading samples of each class rather than assumed from the numbers
alone: **0 = Negative, 1 = Neutral, 2 = Positive**. Reasonably balanced (1,105 / 887 / 1,048), no
severe class-imbalance handling needed later.

## Worked Examples

Several fixed row indices are used throughout, each picked to show one specific phenomenon found
during the Step 0 investigation below, rather than reusing a single example that happens not to
show most of them:

- `DEMO_IDX = 0` — the plain running example ("máy dùng hay bị đơ máy"), already clean, threaded
  through every step to show what *doesn't* change.
- `GLUE_REAL_IDX = 210` / `GLUE_BRAND_IDX = 352` — a genuine missing-space glue versus a brand name
  that looks like one.
- `PUNCT_IDX = 786` — one of the few rows that actually has punctuation, to make Step 2 visible.
- `VS_IDX = 155` — a clean `vs` abbreviation example.
- `STOPWORD_PHRASE_IDX = 1650` — a row containing a multi-word stopword phrase.

In [3]:
DEMO_IDX = 0
GLUE_REAL_IDX = 210
GLUE_BRAND_IDX = 352
PUNCT_IDX = 786
VS_IDX = 155
STOPWORD_PHRASE_IDX = 1650

print("DEMO_IDX:            ", df.loc[DEMO_IDX, "comment"])
print("GLUE_REAL_IDX:        ", df.loc[GLUE_REAL_IDX, "comment"][:100])
print("GLUE_BRAND_IDX:       ", df.loc[GLUE_BRAND_IDX, "comment"][:100])
print("PUNCT_IDX:            ", df.loc[PUNCT_IDX, "comment"][:100])
print("VS_IDX:               ", df.loc[VS_IDX, "comment"])
print("STOPWORD_PHRASE_IDX:  ", df.loc[STOPWORD_PHRASE_IDX, "comment"])

## Step 0 · Investigating What Noise This Corpus Actually Has

The task named four categories to check: abbreviations (viết tắt), missing diacritics (không
dấu), emoji/icons, and glued-together words (dính chữ). Checking each on the real data, with real
counts and real examples, before deciding what's worth fixing — not assuming the news-corpus or
AIVIVN-corpus findings just carry over, since this is a different (and visibly cleaner) source.

### URLs and HTML

In [4]:
url_matches = re.findall(r"https?://\S+", " ".join(df["comment"]))
print(f"URL matches: {len(url_matches)} (out of {len(df):,} reviews)")

None. Nothing to strip here.

### Icons / Emoji

Same unicode-range check as the AIVIVN investigation in this project's earlier draft.

In [5]:
EMOJI_PATTERN = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]")
emoji_rows = df["comment"].apply(lambda t: bool(EMOJI_PATTERN.search(t)))
print(f"Rows with at least one emoji: {emoji_rows.sum()} / {len(df)} ({emoji_rows.mean():.1%})")
print(df.loc[emoji_rows, "comment"].head(3).tolist())

Only ~0.2% of rows (vs. ~7% in the AIVIVN corpus checked earlier). **Decision: same as before —
leave unicode emoji untouched by cleaning; they still won't become model features later since
`TfidfVectorizer`'s default token pattern doesn't treat them as word characters, and at this
frequency it isn't worth building a custom tokenizer just for them.**

### Glued-Together Words (dính chữ)

Checked for a lowercase letter directly followed by an uppercase letter with **no space between
them** — using Python's own Unicode-aware `str.islower()` / `str.isupper()` per character, not a
hand-written character-range regex (a first attempt with a `[a-zà-ỹ]`-style range matched 96% of
rows — that Unicode range spans several unrelated blocks between `à` and `ỹ`, not just Vietnamese
letters, so it's not a valid technique here).

In [6]:
def has_glue(text: str) -> bool:
    return any(a.isalpha() and b.isalpha() and a.islower() and b.isupper() for a, b in zip(text, text[1:]))

glue_rows = df["comment"].apply(has_glue)
print(f"Rows with a lower->upper glue: {glue_rows.sum()} / {len(df)} ({glue_rows.mean():.1%})")

In [7]:
print("Genuine missing-space glue:")
print(" ", df.loc[GLUE_REAL_IDX, "comment"][:120])
print()
print("Same pattern, but it's a brand name, not a missing space:")
print(" ", df.loc[GLUE_BRAND_IDX, "comment"][:120])

Row 210 has `hayXài` — genuinely two words run together with the sentence break dropped
(`hay. Xài` → `hay` + `Xài`, no space, no period). Row 352 has `tikiNow` — not a bug at all, it's
Tiki's own delivery-service brand name, written in its actual casing. Both match the exact same
lower→upper character pattern, and nothing at the character level distinguishes them (`tikiNow`'s
lowercase run before the capital is `tiki`, 4 letters; `hayXài`'s is `hay`, 3 letters — length
doesn't separate genuine glue from brand casing either, same finding as the earlier AIVIVN
investigation with `iPhone`/`ZenBook`/`LinkCard`).

**Decision: not auto-fixed.** At 2.4% of rows (72/3,040) and with no reliable way to tell a missing
space from a brand name at the character level, an automatic "insert a space here" rule would just
as often break real product/service names as fix real typos. Left as-is; documented here instead
of silently dropped.

### Missing Diacritics (không dấu)

Rows that contain letters but zero Vietnamese diacritic marks — meaning the review was typed
without accents entirely (`san pham` instead of `sản phẩm`), not that it happens to be a short word
that never needed one.

In [8]:
VIET_DIACRITICS = set("àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ")
VIET_DIACRITICS |= {c.upper() for c in VIET_DIACRITICS}

has_letters = df["comment"].apply(lambda t: any(c.isalpha() for c in t))
has_diacritics = df["comment"].apply(lambda t: any(c in VIET_DIACRITICS for c in t))
no_diacritics = has_letters & ~has_diacritics
print(f"No-diacritic rows: {no_diacritics.sum()} / {has_letters.sum()} ({no_diacritics.sum()/has_letters.sum():.1%})")
print(df.loc[no_diacritics, "comment"].tolist())

Only 5 rows (0.2%) — much cleaner than the AIVIVN corpus (3.4% there). Two of the five ("Ngon",
"Perfect") aren't even really missing anything; they're just words that don't happen to carry a
diacritic mark to begin with, or are English. **Decision: not fixed — same reasoning as before**
(reliably restoring dropped diacritics is its own hard NLP problem, a dedicated
restoration model, not a cleaning rule), but at this incidence it barely matters here anyway.

### Abbreviations / Teencode (viết tắt)

Same methodology as the news-preprocessing project: count candidate short tokens, then read real
context for the ambiguous ones before deciding to expand them — a candidate is only expanded if
its real usage in *this* corpus is consistently one meaning, not assumed from the abbreviation
alone.

In [9]:
VI_CHARS = "a-z0-9àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩịòóỏõọôồốổỗộơờớởỡợùúủũụưừứửữựỳýỷỹỵđ"
lower_comments = df["comment"].str.lower()
tokens_per_row = lower_comments.apply(lambda t: re.findall(f"[{VI_CHARS}]+", t))

from collections import Counter
token_counts = Counter(tok for row in tokens_per_row for tok in row)

candidates = ["k", "ko", "kg", "dc", "đc", "sp", "vs", "mn", "sz", "ship", "ok", "oke"]
for tok in candidates:
    print(f"{tok!r}: {token_counts.get(tok, 0)}")

Compare to the AIVIVN corpus checked in this project's earlier draft, where `ko` alone appeared
over 2,000 times: here almost every teencode candidate is at or near zero. This corpus is written
in much more standard Vietnamese. `ok`/`oke` are common (229 combined) but those are just the
English loanword "ok" used as-is in casual Vietnamese, not an abbreviation of a Vietnamese word —
nothing to expand. `ship` (18) is the same kind of loanword ("giao hàng"/delivery used
interchangeably with the English word in e-commerce slang) — left as-is for the same reason.

The one candidate worth checking properly is `vs`, since in the AIVIVN corpus it consistently meant
"với" (with), not English "versus":

In [10]:
print(df.loc[VS_IDX, "comment"])

Same finding as before: `vs` → `với`. **Decision: a one-entry teencode map for this corpus** —
everything else checked is either genuinely rare enough to ignore or is a loanword that shouldn't
be translated away.

In [11]:
TEENCODE_MAP = {
    "vs": "với",
}

### A Fifth Finding, Not On The Original List: Syllable-Splitting A Real Compound Word

Scanning short tokens above also turned up something that isn't teencode at all. Words like `uy`
in `uy tín` (trustworthy) are single syllables that only mean something as *half of* a two-syllable
compound word — the tokenizer used matters here more than any cleaning rule: `nltk.word_tokenize`
splits `uy tín` into two unrelated tokens `uy` and `tín`, while `underthesea.word_tokenize` (used
below) recognizes it as one word and joins it `uy_tín`. This is the exact word-segmentation-versus-
n-gram distinction already worked out in
`260106_Scikit-learnTextFeatureExtraction/Personal Note.md` — not re-derived here, just confirmed
to matter for this corpus too, which is the actual reason this project switches to `underthesea`
for tokenization instead of reusing `nltk.word_tokenize` from the earlier two projects.

### Step 0 Summary

| Category | Found | Fixed? | Why |
|---|---|---|---|
| URLs / HTML | 0 rows | n/a | not present |
| Emoji / icons | 0.2% of rows | No | too rare here to matter, and wouldn't become a TF-IDF feature anyway |
| Glued words | 2.4% of rows | No | indistinguishable from real brand-name casing (`tikiNow`) at the character level |
| Missing diacritics | 0.2% of rows | No | needs a dedicated restoration model; rare here regardless |
| Abbreviations | 1 confirmed (`vs`) | Yes | `TEENCODE_MAP`, context-checked, not assumed |
| Syllable-split compounds | pervasive (any multi-syllable word) | Yes | switching to `underthesea` for tokenization |

## Step 1 · Unicode Normalization (NFC)

In [12]:
def normalize_unicode(text: str) -> str:
    return unicodedata.normalize("NFC", str(text))

df["comment"] = df["comment"].apply(normalize_unicode)

## Step 2 · Removing Noise

Only what Step 0 actually found worth stripping: URLs (none, but kept for safety/reuse) and
punctuation/digits. Most rows in this corpus have no punctuation at all (see `DEMO_IDX`), but a few
do (`PUNCT_IDX`), so the effect is visible on that row and not on the others.

In [13]:
EXTRA_PUNCTUATION = "\u201c\u201d\u2018\u2019\u2026\u2013\u2014"

def clean_text(text: str) -> str:
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # URLs
    text = re.sub(r"[0-9]+", " ", text)               # digits
    text = re.sub(f"[{re.escape(string.punctuation)}{EXTRA_PUNCTUATION}]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

before = df.loc[PUNCT_IDX, "comment"]
df["comment"] = df["comment"].apply(clean_text)
after = df.loc[PUNCT_IDX, "comment"]
print("Before:", before)
print("After: ", after)
print()
print("DEMO_IDX unaffected:", df.loc[DEMO_IDX, "comment"])
print("Still glued (not fixed here, by design):", df.loc[GLUE_REAL_IDX, "comment"][:80])

## Step 3 · Lowercasing

In [14]:
before = df.loc[DEMO_IDX, "comment"]
df["comment"] = df["comment"].str.lower()
after = df.loc[DEMO_IDX, "comment"]
print("Before:", before)
print("After: ", after)

## Step 4 · Tokenization With `underthesea`

Benchmarking on a random sample first, same caution used in `260106_Word2Vec` — `underthesea` took
about 2 hours on the 184K-article news corpus, so its cost has to be checked before committing to a
full run, not assumed to be fine just because it worked at a different scale.

In [15]:
sample = df["comment"].sample(200, random_state=0).tolist()

t0 = time.time()
for text in sample:
    word_tokenize(text, format="text")
elapsed = time.time() - t0
per_item = elapsed / len(sample)

print(f"{len(sample)} items in {elapsed:.2f}s -> {per_item*1000:.2f} ms/item")
print(f"estimated full corpus ({len(df):,} rows): {per_item * len(df):.1f}s")

A few seconds for the full corpus — reviews are short (a sentence or two), nothing like 184K full
news articles. Safe to just run it on everything.

In [16]:
before = df.loc[DEMO_IDX, "comment"]
df["tokens"] = df["comment"].apply(lambda t: word_tokenize(t, format="text").split())
after = df.loc[DEMO_IDX, "tokens"]
print("Before:", before)
print("After: ", after)

Compound words actually join now, unlike the syllable-level `nltk` output used previously:

In [17]:
sample_row = df[df["comment"].str.contains("sản phẩm", na=False)].index[0]
print("Before:", df.loc[sample_row, "comment"][:80])
print("After: ", df.loc[sample_row, "tokens"][:12])

## Step 5 · Abbreviation Expansion

Applied on the token list (exact match only), not as a string find-and-replace — this avoids ever
matching an abbreviation as a substring of a longer, unrelated word.

In [18]:
def expand_teencode(tokens: list[str]) -> list[str]:
    return [TEENCODE_MAP.get(t, t) for t in tokens]

before = df.loc[VS_IDX, "tokens"] if VS_IDX in df.index else None
df["tokens"] = df["tokens"].apply(expand_teencode)
print("Before:", word_tokenize(df.loc[VS_IDX, "comment"], format="text").split())
print("After: ", df.loc[VS_IDX, "tokens"])

## Step 6 · Stopword Removal — Investigated, Then Decided Against

The same Vietnamese stopword list as the other projects
(`heeraldedhia/stop-words-in-28-languages`, `vietnamese.txt`) is available and, same finding as
before, needs its multi-word entries (`"nói chung"`, 81% of the list) normalized to their
underscore-joined form before they can match `underthesea`'s compound tokens (`nói_chung`) — a
real technical gotcha worth keeping on record even though the filtering itself isn't applied below.

**Why not applied**: `MultinomialNB` is traditionally paired with raw Bag-of-Words counts, not
TF-IDF (the actual reason this project ends up comparing NB+BoW, NB+TF-IDF, and SVM+TF-IDF head to
head in Steps 2-3, rather than assuming one vectorizer for everything) — and for a *sentiment*
task specifically, generic stopword lists remove exactly the words that carry the sentiment signal.
Checked, not assumed:

In [19]:
stopwords_path = kagglehub.dataset_download("heeraldedhia/stop-words-in-28-languages", "vietnamese.txt")
with open(stopwords_path, encoding="utf-8") as f:
    raw_stopwords = {line.strip() for line in f if line.strip()}

vi_stopwords = set(raw_stopwords)
vi_stopwords |= {w.replace(" ", "_") for w in raw_stopwords if " " in w}
print(f"{len(raw_stopwords)} raw entries -> {len(vi_stopwords)} after adding underscore-joined forms")

for word in ["không", "chưa", "rất", "quá", "tốt"]:
    print(f"{word!r} in stopword list: {word in raw_stopwords}")

In [20]:
# không (not), chưa (not yet), rất (very), quá (too/very) are all in the list -- exactly the
# negation and intensifier words that flip or scale a review's sentiment. Removing them would
# throw away the signal this whole project is trying to classify, e.g. "không tốt" (not good)
# would lose "không" and read as just "tốt" (good) -- the opposite of the actual meaning.
# Decision: vi_stopwords is built above for the record, but remove_stopwords() is not called on
# df["tokens"] -- stopwords stay in the corpus that gets exported below.
print("Example of what would be destroyed:", [t for t in ["không", "tốt"] if t in vi_stopwords])

`không` would have been silently dropped by a generic stopword list applied without checking what
it actually contains for this task — this is the concrete version of the abstract "NB wants BoW,
not TF-IDF" comment that started this whole detour: a vectorization/modeling choice can't be
picked from habit, it has to be checked against what the task actually needs.

## Exporting

No train/test split here. Steps 2-4 compare Naive Bayes (Bag-of-Words and TF-IDF) and SVM
(TF-IDF) with `StratifiedKFold` cross-validation instead of a single held-out split — every row
gets used for both training and evaluation across the folds, which gives a more stable comparison
between three methods on a dataset this size (3,040 rows) than one lucky/unlucky 80/20 split would.
`StratifiedKFold(..., random_state=42)` is re-created with the same seed in every notebook that
uses it, so all three methods are compared on the exact same folds without needing to persist fold
indices to disk.

Vectorization (`CountVectorizer` / `TfidfVectorizer`) is *not* fit here either, on purpose — fitting
it on the whole corpus before splitting into folds would leak each fold's own vocabulary/IDF
statistics into the folds that are supposed to be held out from it. It gets fit fresh inside each
fold, in Steps 2-3, on that fold's training portion only. What's exported here is exactly the
dataset-level work described at the top of this notebook: cleaned, normalized, tokenized (real
word segmentation via `underthesea`), teencode-expanded text — stopwords intentionally still in.

In [21]:
df["clean_comment"] = df["tokens"].apply(lambda toks: " ".join(toks))

export_df = df[["clean_comment", "label"]].reset_index(drop=True)
print(export_df.shape)
print(export_df["label"].value_counts(normalize=True).sort_index())
export_df.head()

## Save

In [22]:
out_path = os.path.join(PROCESSED_DIR, "reviews.parquet")
export_df.to_parquet(out_path, index=False)
print(f"Saved {len(export_df)} rows -> {out_path}")